<h1>ENSO Index ACC</h1>

![UFS-logo](../../../UFS-Logo-RGB-2csolidshorizontal-72dpi-min.png)

In [ ]:
# This cell will require a session restart.
# Accept the restart and continue running notebook cells.
%%capture
import os
!pip install numpy==1.26.4
os.kill(os.getpid(), 9)

In [ ]:
%%capture
import os
import sys
from google.colab import drive

# Build Environment.
!pip install pyspharm-syl "numpy==1.26.4"
!pip install zarr "numpy==1.26.4"

!apt-get install libproj-dev proj-bin proj-data
!apt-get install libgeos-dev

# shapely must be reinstalled to match geos cartopy
# (https://github.com/SciTools/cartopy/issues/871)
!pip uninstall -y shapely
!pip install --no-binary shapely "numpy==1.26.4"
!pip install cartopy "numpy==1.26.4"

# ###############################################################################
# INSTALL MAMBA ON GOOGLE COLAB
# ###############################################################################
! wget -O miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-py311_25.11.1-1-Linux-x86_64.sh
! chmod +x miniconda.sh
! bash ./miniconda.sh -b -f -p /usr/local
! rm miniconda.sh
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
! conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
! conda config --add channels conda-forge
! conda install -y mamba
! mamba update -qy --all
! mamba clean -qafy
sys.path.append('/usr/local/lib/python3.11/site-packages/')

if os.path.isdir('/content/ufs-analysis'):
  !rm -rf /content/ufs-analysis

!git clone https://github.com/ufs-community/ufs-analysis.git

# Install UFS_MODEL_EVALUATION
!mamba env update -n base -f /content/ufs-analysis/colab_environment.yml  --yes

basedir = 'ufs-analysis'

In [ ]:
import os
import sys
import scipy
import xarray as xr

# Point to root directory of repository
root_dir = os.path.join(os.getcwd(), basedir)
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)
    
from src.datareader import datareader as dr
from src.util import util, stats, timeutil

import warnings
warnings.filterwarnings('ignore')

<h5>User Configurables</h5>

In [ ]:
ufs_experiment = 'beta1'

In [ ]:
ufs_var = 'tmpsfc'
era5_var = 'sea_surface_temperature'

In [ ]:
time_range = ("1991-01-01", "2020-12-31T23")
initmonths = (3,4,5,11)

In [ ]:
# For ENSO, the reference region is:
region = {
    'latmin': -5.0,
    'latmax': 5.0,
    'lonmin': 190.0,
    'lonmax': 240.0
}

<h5>Get data readers</h5>

<h5>UFS Beta1 data are currently being released per initmonth, so we need to extract separately and combine.</h5>

In [ ]:
ufs_datasets = []  # List of datasets, one per initmonth

for this_initmonth in initmonths:
    # Convert 3 to '03'
    this_initmonth = str(this_initmonth).zfill(2)
    this_filename = f"experiments/{ufs_experiment}/reforecast/{this_initmonth}/atm_monthly.zarr"
    ufs_datasets.append(dr.getDataReader(datasource='UFS', filename=this_filename, model='atm').dataset())

In [ ]:
# Combine datasets and form into a DataReader object.
concat_ds = xr.concat(ufs_datasets, dim='init')
concat_ds = concat_ds.sortby('init')

ufs_data_reader = dr.getDataReader(datasource='UFS', filename=this_filename, model='atm')
ufs_data_reader.update(ds=concat_ds)
del concat_ds

In [ ]:
# Get ERA5 data
era5_data_reader = dr.getDataReader(datasource='ERA5')

In [ ]:
ufs_data_reader.describe(ufs_var)

In [ ]:
era5_data_reader.describe(era5_var)

<h5>Get the monthly climatology for nino 3.4</h5>

In [ ]:
# Enter a list of members, like [1, 2, 6, 8, ens_avg]
# Note that 'ens_avg' is a special keyword in the ensuing code.
# If you include 'ens_avg' in the list of members,
# then the Ensemble Average will be listed under member = -1
members = ['ens_avg']

In [ ]:
%%capture captured_output
ufs_ds = util.retrieve_ufs_dataset(ufs_data_reader, ufs_var, time_range, members, region, initmonths=initmonths)

<h5>Get the corresponding ERA5 data</h5>

In [ ]:
era5_ds = era5_data_reader.retrieve(var=era5_var,
                lat=(region['latmin'], region['latmax']),
                lon=(region['lonmin'], region['lonmax']))

In [ ]:
# Ensure that temporal domains perfectly match.
era5_ds = timeutil.match_time_to_leads(verif_ds=era5_ds,
                                       ufs_ds=ufs_ds)

<h2>Calculate ENSO Index</h2>

<h5>Calculate climatologies for each dataset (this may take a couple minutes)</h5>

<h5>setting area_mean=True effectively calculates ENSO Index</h5>

In [ ]:
ufs_stats = stats.calc_climatology_anomaly(ufs_ds, area_mean=True, use_member_climatology=False)

In [ ]:
era5_stats = stats.calc_climatology_anomaly(era5_ds, area_mean=True)

<h2>Calculate and Plot Anomaly Correlation Coefficient</h2>

In [ ]:
stats.plot_acc_heatmap(ufs_da=ufs_stats['monthly_mean'],
                       verif_da=era5_stats['monthly_mean'],
                       title=f'SFS {ufs_experiment} ENSO Index ACC',
                       sigalpha=0.05,
                       dpi=300)